# Comparing processed somatic variants with ClinVar unfiltered for pathogenicity

## Process ClinVar without PLP filter

In [ ]:
# 1) import modules
import os
import re
import json
import pandas as pd
import scipy
import time
import requests
import hashlib
import csv
import random
from collections import defaultdict
import numpy as np    
import statsmodels.api as sm   
import seaborn as sns
from scipy.signal import find_peaks
import matplotlib.pyplot as plt

In [ ]:
start_time_block2=time.asctime(time.localtime())
print(start_time_block2)


allTSGdfcounts=pd.DataFrame()
parentpath="/Users/suhasinilulla/Library/CloudStorage/Box-Box/PhD_BCM/Lulla_Plon_Lab/High_priority_gene_list/GL_S_Pipeline_1st_pass_47TSGs_May2023"
os.chdir(parentpath)


#read file with folder names and MANE Select transcript identifiers (or MANE Plus Clinical in the case of SMARCA4)
TranscriptID= pd.read_csv("TSGfolder_MANESelect_names_aalength_GeneIDs_6-13-24.txt", sep="\t")


for index, row in TranscriptID.iterrows():
    
    # Loop through each folder name
    folder_name=TranscriptID.loc[index]["Gene Symbol"]
    #if folder name exists, list all files in that folder then change directory to that folder
    if os.path.exists(folder_name) and os.path.isdir(folder_name):
        filelist=os.listdir(folder_name)
        os.chdir(folder_name)
        print(folder_name)
        print("Current directory:", os.getcwd())
        os.chdir("Python_coderun_6-13-24_clinvar_extract_and_process_again")
        
        # read extracted file with the column names as defined here
        inputfile_clinvar_colnames = ["GeneSymbol", "GeneID", "VariationID", "VariationName", "VariationType","NumberOfSubmissions", "NumberOfSubmitters","GermlineClassificationNumberOfSubmissions","GermlineClassificationNumberOfSubmitters", "ReviewStatus", "GL_first_Description", "Explanation", "SubmissionList", "HGVS", "Assembly"]
        #BRCA1 and BRCA2 run on 6-17:
        if (folder_name=="BRCA1") or (folder_name=="BRCA2"):
            clinvar_rawdata = pd.read_csv("ClinVarAPI_xtract_update_6-17-24.txt", sep="\t", names=inputfile_clinvar_colnames, header=None, usecols=[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14])
        else:
            clinvar_rawdata = pd.read_csv("ClinVarAPI_xtract_update_6-14-24.txt", sep="\t", names=inputfile_clinvar_colnames, header=None, usecols=[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14])
        print(len(clinvar_rawdata))
        One_allclinvar=len(clinvar_rawdata)
        
        #filter variation type
        Variation_type_toexclude = ["Complex", "Translocation", "Variation", "fusion", "protein only", "Haplotype", "copy number gain", "copy number loss"]
        clinvar_rawdata_vartypefilter = clinvar_rawdata[~clinvar_rawdata['VariationType'].isin(Variation_type_toexclude)]
        print(len(clinvar_rawdata_vartypefilter))
        Two_clinvarvartype=len(clinvar_rawdata_vartypefilter)
        
        
        #filter conflicting variants:
        clinvar_rawdata_vartype_reviewstatus_conflict= clinvar_rawdata_vartypefilter[clinvar_rawdata_vartypefilter['GL_first_Description'].str.contains("Conflicting classifications of pathogenicity")]
        #rename colnames for conflicting df to match with p/lp so that can concate conflicting variants scoring>75 with p/lp df
        clinvar_rawdata_vartype_reviewstatus_conflict_rename=clinvar_rawdata_vartype_reviewstatus_conflict.rename(columns={"HGVS":"HGVSc", "Assembly":"HGVSg"})
        
        clinvar_rawdata_vartype_reviewstatus_PLP=clinvar_rawdata_vartypefilter.drop(clinvar_rawdata_vartype_reviewstatus_conflict.index)
        #rename colnames for PLP df (since these variants dont have an 'Explanation' column- that col is present only for conflicting variants, but eutils xtract mushes all info together and doesnt demarcate sep cols)
        clinvar_rawdata_vartype_reviewstatus_PLP_rename=clinvar_rawdata_vartype_reviewstatus_PLP.rename(columns={"Explanation":"PLP_SubmissionList", "SubmissionList":"PLP_HGVSc", "HGVS":"PLP_HGVSg"})
        #have to do it in 2 steps since cant have 2 cols with same name
        clinvar_rawdata_vartype_reviewstatus_PLP_rename=clinvar_rawdata_vartype_reviewstatus_PLP_rename.rename(columns={"PLP_SubmissionList":"SubmissionList", "PLP_HGVSc":"HGVSc", "PLP_HGVSg":"HGVSg"})

        #calculate score for conflicting
        #adding col for score
        clinvar_rawdata_vartype_reviewstatus_conflict_rename.insert(1,"Score","NaN")
        for index, row in clinvar_rawdata_vartype_reviewstatus_conflict_rename.iterrows():
            rowindex=clinvar_rawdata_vartype_reviewstatus_conflict_rename.index.get_loc(index)
            colindex=clinvar_rawdata_vartype_reviewstatus_conflict_rename.columns.get_loc('Score')
            
            colindex_interpretation=clinvar_rawdata_vartype_reviewstatus_conflict_rename.columns.get_loc('GL_first_Description')
            
            #initialize counts to 0 for every row (aka variant)
            count_vus=count_p=count_lp=count_b=count_lb=score_vus=score_p=score_lp=score_b=score_lb=int(0)
            #print(clinvar_rawdata_vartype_reviewstatus_conflict_rename.loc[index]["Explanation"])
            explanation=str(clinvar_rawdata_vartype_reviewstatus_conflict_rename.loc[index]["Explanation"]).split("; ")
            for item in explanation:
                if item.startswith("Uncertain significance"):
                    valueip=item.split("(")
                    valueip2=valueip[1].split(")")
                    count_vus=count_vus+int(valueip2[0])
                    #score_vus=score_vus+(50*int(count_vus))
                    #print(count_vus, score_vus)
                else:
                    if item.startswith("Pathogenic"):
                        valueip=item.split("(")
                        valueip2=valueip[1].split(")")
                        count_p=count_p+int(valueip2[0])
                        #score_p=score_p+(99.5*int(count_p))
                        #print(count_p, score_p)

                    else:
                        if item.startswith("Likely pathogenic"):
                            valueip=item.split("(")
                            valueip2=valueip[1].split(")")
                            count_lp=valueip2[0]
                            #score_lp=score_lp+(95*int(count_lp))
                            #print(score_lp)

                        else:
                            if item.startswith("Likely benign"):
                                valueip=item.split("(")
                                valueip2=valueip[1].split(")")
                                count_lb=valueip2[0]
                                #score_lb=score_lb+(5*int(count_lb))
                                #print(score_lb)

                            else:
                                if item.startswith("Benign"):
                                    valueip=item.split("(")
                                    valueip2=valueip[1].split(")")
                                    count_b=valueip2[0]
                                    #score_b=score_b+(0.05*int(count_b))
                                    #print(score_b)
                                    
            score_vus=(50*int(count_vus))
            score_p=(99.5*int(count_p))
            score_lp=(95*int(count_lp))
            score_lb=(5*int(count_lb))
            score_b=(0.05*int(count_b))
            score=(score_vus+score_p+score_lp+score_b+score_lb)/(int(count_vus)+int(count_p)+int(count_lp)+int(count_b)+int(count_lb))
        
            clinvar_rawdata_vartype_reviewstatus_conflict_rename.iloc[rowindex,colindex]=score
            if (score>=75):
                clinvar_rawdata_vartype_reviewstatus_conflict_rename.iloc[rowindex,colindex_interpretation]="Conflicting interpretations of pathogenicity greater than or equal to 75"
            else:
                clinvar_rawdata_vartype_reviewstatus_conflict_rename.iloc[rowindex,colindex_interpretation]="Conflicting interpretations of pathogenicity less than 75"

   
        
        #concat PLP and conflicting:
        clinvar_rawdata_vartype_reviewstatus_PLPconflict_data = [clinvar_rawdata_vartype_reviewstatus_PLP_rename, clinvar_rawdata_vartype_reviewstatus_conflict_rename]
        clinvar_rawdata_vartype_reviewstatus_PLPconflict = pd.concat(clinvar_rawdata_vartype_reviewstatus_PLPconflict_data)
        TwoA_QC=len(clinvar_rawdata_vartype_reviewstatus_PLPconflict)
        print(TwoA_QC)
       
    
        clinvar_rawdata_vartype_reviewstatus_PLPconflict_origin_filter=clinvar_rawdata_vartype_reviewstatus_PLPconflict[clinvar_rawdata_vartype_reviewstatus_PLPconflict['SubmissionList'].str.contains("germline|de novo|maternal|paternal|uniparental|biparental|inherited")==True]
        TwoB_Originfilter=len(clinvar_rawdata_vartype_reviewstatus_PLPconflict_origin_filter)
        print(TwoB_Originfilter)
        
        #filter 1 star review status
        Review_status_toexclude = ["no assertion criteria provided", "no assertion provided", "no assertion for the individual variant"]
        clinvar_rawdata_vartype_reviewstatus_filter = clinvar_rawdata_vartype_reviewstatus_PLPconflict_origin_filter[~clinvar_rawdata_vartype_reviewstatus_PLPconflict_origin_filter['ReviewStatus'].isin(Review_status_toexclude)]
        print(len(clinvar_rawdata_vartype_reviewstatus_filter))
        Three_clinvarreviewstatus=len(clinvar_rawdata_vartype_reviewstatus_filter)
        
        
        #filter PLP
        #PLP = ["Pathogenic", "Likely pathogenic", "Pathogenic/Likely pathogenic"]
        #clinvar_rawdata_vartype_reviewstatus_PLP= clinvar_rawdata_vartype_reviewstatus_filter[clinvar_rawdata_vartype_reviewstatus_filter['GL_first_Description'].str.contains("Pathogenic|Likely pathogenic|Pathogenic/Likely pathogenic")]
        #rename colnames for PLP df (since these variants dont have an 'Explanation' column- that col is present only for conflicting variants, but eutils xtract mushes all info together and doesnt demarcate sep cols)
        #print(len(clinvar_rawdata_vartype_reviewstatus_PLP))
        #Four_clinvarPLP=len(clinvar_rawdata_vartype_reviewstatus_PLP)
        
        Five_clinvarconflictall=len(clinvar_rawdata_vartype_reviewstatus_conflict_rename)
        clinvar_rawdata_vartype_reviewstatus_PLPconflict=clinvar_rawdata_vartype_reviewstatus_filter
        
        #filter for score greater than or equal to 75
        #clinvar_rawdata_vartype_reviewstatus_conflict_gte75 = clinvar_rawdata_vartype_reviewstatus_conflict_rename[(clinvar_rawdata_vartype_reviewstatus_conflict_rename["Score"] >= 75)]
        #Six_clinvarconflictgte75=len(clinvar_rawdata_vartype_reviewstatus_conflict_gte75)
        
        
        #Occurrence calculation counting only 1 germline term per submitter (even if they had multiple submissions), if it exists- if not then 0 and var filtered out (in the case of somatic or unknown only submissions)
        clinvar_rawdata_vartype_reviewstatus_PLPconflict.insert(1,"Occurrence","NaN")
        for index,row in clinvar_rawdata_vartype_reviewstatus_PLPconflict.iterrows():
            rowindex=clinvar_rawdata_vartype_reviewstatus_PLPconflict.index.get_loc(index)
            colindex=clinvar_rawdata_vartype_reviewstatus_PLPconflict.columns.get_loc('Occurrence')
    
            originpacked=str(clinvar_rawdata_vartype_reviewstatus_PLPconflict.loc[index]["SubmissionList"]).split("/")
            occurrence=0
            submissionlist=[]
            output_dict = {}
            for item in originpacked:
                submissiondetails=str(item).split("|")
                #print(submissiondetails)
                #itemdict=pd.DataFrame({k: v for k, *v in item}).to_dict('records')
                #print(itemdict)
                submissionlist.append(submissiondetails)
            submissionlist=submissionlist[:-1]
            for key, *values in submissionlist:
                if key not in output_dict:
                    output_dict[key] = [key]
                output_dict[key].extend(values)

            for item2 in output_dict.values():
                #print(item2)
                if ('germline' in item2) or ('de novo' in item2) or ('maternal' in item2) or ('paternal' in item2) or ('uniparental' in item2) or ('biparental' in item2) or ('inherited' in item2):
                    occurrence=occurrence+1
            clinvar_rawdata_vartype_reviewstatus_PLPconflict.iloc[rowindex,colindex]=occurrence
            
            # hashed code below so it wont execute --6-14-24
            #print(clinvar_rawdata_vartype_reviewstatus_PLPconflict.loc[index]["NumberOfSubmissions"], clinvar_rawdata_vartype_reviewstatus_PLPconflict.loc[index]["NumberOfSubmitters"], clinvar_rawdata_vartype_reviewstatus_PLPconflict.loc[index]["GermlineClassificationNumberOfSubmissions"], clinvar_rawdata_vartype_reviewstatus_PLPconflict.loc[index]["GermlineClassificationNumberOfSubmitters"], occurrence)

        clinvar_rawdata_vartype_reviewstatus_PLPconflict_occurrencegt0=clinvar_rawdata_vartype_reviewstatus_PLPconflict[clinvar_rawdata_vartype_reviewstatus_PLPconflict["Occurrence"]>0]
        clinvar_rawdata_vartype_reviewstatus_PLPconflict_origin_filter=clinvar_rawdata_vartype_reviewstatus_PLPconflict_occurrencegt0
        print(len(clinvar_rawdata_vartype_reviewstatus_PLPconflict_origin_filter))
        Eight_clinvarorigin=len(clinvar_rawdata_vartype_reviewstatus_PLPconflict_origin_filter)
        
        print(clinvar_rawdata_vartype_reviewstatus_PLPconflict_origin_filter["Occurrence"].value_counts())
        print(clinvar_rawdata_vartype_reviewstatus_PLPconflict_origin_filter["Occurrence"].sum())
        #clinvar_rawdata_vartype_reviewstatus_PLPconflict_origin_filter.to_csv("Python_clinvar_rawdata_vartype_reviewstatus_PLPconflict_origin_filter_forfutureQC_6-24-24.txt", sep="\t")
        
        #HGVSc filter: drop blanks (meaning no ref or alt info for the variant)
        clinvar_rawdata_vartype_reviewstatus_PLPconflict_origin_filter=clinvar_rawdata_vartype_reviewstatus_PLPconflict_origin_filter.dropna(subset=["HGVSc"])
        #HGVSc filter drop variants with '?' in HGVSc:
        clinvar_rawdata_vartype_reviewstatus_PLPconflict_origin_filter.insert(1,"HGVScfilter","NaN")
        for index, row in clinvar_rawdata_vartype_reviewstatus_PLPconflict_origin_filter.iterrows():
            rowindex=clinvar_rawdata_vartype_reviewstatus_PLPconflict_origin_filter.index.get_loc(index)
            colindex=clinvar_rawdata_vartype_reviewstatus_PLPconflict_origin_filter.columns.get_loc('HGVScfilter')
            if "?" not in clinvar_rawdata_vartype_reviewstatus_PLPconflict_origin_filter.loc[index]['HGVSc']:
                clinvar_rawdata_vartype_reviewstatus_PLPconflict_origin_filter.iloc[rowindex,colindex]="has no question mark"
        clinvar_rawdata_filter_PLPconflict_HGVScfilter=clinvar_rawdata_vartype_reviewstatus_PLPconflict_origin_filter[clinvar_rawdata_vartype_reviewstatus_PLPconflict_origin_filter["HGVScfilter"].str.contains("has no question mark")==True]
        
        #added these 2 codes in 6-21-24 update to see if any vars are on GRCh37 assembly since I removed that filter bc the new extraction code does not ask for vars on GRCh37
        #display(clinvar_rawdata_filter_PLPconflict_HGVScfilter["Assembly"].value_counts())
        #display(clinvar_rawdata_filter_PLPconflict_HGVScfilter["HGVSg"].value_counts())
        
        display(clinvar_rawdata_filter_PLPconflict_HGVScfilter["HGVSc"].value_counts())
        Nine_HGVScfilter=len(clinvar_rawdata_filter_PLPconflict_HGVScfilter)
        print(Nine_HGVScfilter)
        
        ##added code block on 6-24-24 to account for vars with sq brackets in HGVSc '[xx]'
        clinvar_rawdata_filter_PLPconflict_HGVScfilter.insert(1,"HGVSc_new","NaN")
        for index, row in clinvar_rawdata_filter_PLPconflict_HGVScfilter.iterrows():
            rowindex=clinvar_rawdata_filter_PLPconflict_HGVScfilter.index.get_loc(index)
            colindex=clinvar_rawdata_filter_PLPconflict_HGVScfilter.columns.get_loc('HGVSc_new')
            if "[" not in clinvar_rawdata_filter_PLPconflict_HGVScfilter.loc[index]['HGVSc']:
                clinvar_rawdata_filter_PLPconflict_HGVScfilter.iloc[rowindex,colindex]= clinvar_rawdata_filter_PLPconflict_HGVScfilter.loc[index]['HGVSc']
            else:
                if pd.isna(clinvar_rawdata_filter_PLPconflict_HGVScfilter.loc[index]['HGVSg']):
                    clinvar_rawdata_filter_PLPconflict_HGVScfilter.iloc[rowindex,colindex]= clinvar_rawdata_filter_PLPconflict_HGVScfilter.loc[index]['HGVSc']
                else:
                    clinvar_rawdata_filter_PLPconflict_HGVScfilter.iloc[rowindex,colindex]= clinvar_rawdata_filter_PLPconflict_HGVScfilter.loc[index]['HGVSg']
               
        
        HGVSc=clinvar_rawdata_filter_PLPconflict_HGVScfilter["HGVSc"]
        clinvar_processed_data_diffhgvsc=clinvar_rawdata_filter_PLPconflict_HGVScfilter[~clinvar_rawdata_filter_PLPconflict_HGVScfilter["HGVSc_new"].isin(HGVSc)]
        #print(clinvar_processed_data_diffhgvsc["HGVSc"], clinvar_processed_data_diffhgvsc["HGVSc_new"])    
        print(len(clinvar_processed_data_diffhgvsc))
        
        
        clinvar_rawdata_filter_PLPconflict_HGVScfilter2=clinvar_rawdata_filter_PLPconflict_HGVScfilter[~clinvar_rawdata_filter_PLPconflict_HGVScfilter["HGVSc_new"].str.contains("\[")]
        clinvar_rawdata_filter_PLPconflict_HGVScfilter2=clinvar_rawdata_filter_PLPconflict_HGVScfilter2[~clinvar_rawdata_filter_PLPconflict_HGVScfilter2["HGVSc_new"].str.contains("GRCh38")]
        
        NineA=len(clinvar_rawdata_filter_PLPconflict_HGVScfilter2)
        print(NineA)
        
        QC_lostvars=clinvar_rawdata_filter_PLPconflict_HGVScfilter.drop(clinvar_rawdata_filter_PLPconflict_HGVScfilter2.index)
        display(QC_lostvars)
        
        clinvar_rawdata_filter_PLPconflict_HGVScfilter=clinvar_rawdata_filter_PLPconflict_HGVScfilter2
        Ten_occurrencetotal=clinvar_rawdata_filter_PLPconflict_HGVScfilter["Occurrence"].sum()
        Eleven_glsubmittertotal=clinvar_rawdata_filter_PLPconflict_HGVScfilter["GermlineClassificationNumberOfSubmitters"].astype(int).sum()
        Twelve_glsubmissiontotal=clinvar_rawdata_filter_PLPconflict_HGVScfilter["GermlineClassificationNumberOfSubmissions"].astype(int).sum()
        Thirteen_submittertotal=clinvar_rawdata_filter_PLPconflict_HGVScfilter["NumberOfSubmitters"].astype(int).sum()
        Fourteen_submissiontotal=clinvar_rawdata_filter_PLPconflict_HGVScfilter["NumberOfSubmissions"].astype(int).sum()
        
        #print HGVSc for annotation
        #clinvar_rawdata_filter_PLPconflict_HGVScfilter["HGVSc_new"].to_csv("Python_clinvar_rawdata_filter_PLPconflict_HGVSctoannotate_6-24-24.txt", sep="\t", index=False, header=None)
        #print processed file for downstream analysis
        clinvar_rawdata_filter_PLPconflict_HGVScfilter.to_csv("Python_clinvar_processed_rawdata_prePLP_7-15-24.txt", sep="\t")

        outputdf=pd.DataFrame({'Gene':[folder_name],'One_allclinvar': [One_allclinvar], 'Two_clinvarvartype':[Two_clinvarvartype], 'TwoA_QC':[TwoA_QC], 'TwoB_Originfilter':[TwoB_Originfilter],'Three_clinvarreviewstatus':[Three_clinvarreviewstatus],'Five_clinvarconflictall':[Five_clinvarconflictall], 'Eight_clinvarorigin':[Eight_clinvarorigin], 'Nine_HGVScfilter':[Nine_HGVScfilter],'NineA_HGVScfiltersqbrackets':[NineA], 'Ten_occurrencetotal':[Ten_occurrencetotal], 'Eleven_glsubmittertotal':[Eleven_glsubmittertotal], 'Twelve_glsubmissiontotal':[Twelve_glsubmissiontotal], 'Thirteen_submittertotal':[Thirteen_submittertotal], 'Fourteen_submissiontotal':[Fourteen_submissiontotal]})
        allTSGdfcounts=pd.concat([allTSGdfcounts,outputdf])
        
        #checkpoint for where the code is while running- print gene name (directory)
        print("Current directory:", os.getcwd())
        #go back to parent directory after completing gene and start over with next gene
        os.chdir(parentpath)
    else:
        print(f"Folder '{folder_name}' not found.")

end_time_block2=time.asctime(time.localtime())
print(end_time_block2)



## Annotate ClinVar PrePLP variant set with CAID

In [ ]:
start_time_block2=time.asctime(time.localtime())
print(start_time_block2)

sharedvarcounts=pd.DataFrame()

parentpath="/Users/suhasinilulla/Library/CloudStorage/Box-Box/PhD_BCM/Lulla_Plon_Lab/High_priority_gene_list/GL_S_Pipeline_1st_pass_47TSGs_May2023"
os.chdir(parentpath)


#read file with folder names and MANE Select transcript identifiers (or MANE Plus Clinical in the case of SMARCA4)
TranscriptID= pd.read_csv("TSGfolder_MANESelect_names_aalength_GeneIDs_6-13-24.txt", sep="\t")

for index, row in TranscriptID.iterrows():
    
    # Loop through each folder name
    folder_name=TranscriptID.loc[index]["Gene Symbol"]
    #if folder name exists, list all files in that folder then change directory to that folder
    if os.path.exists(folder_name) and os.path.isdir(folder_name):
        filelist=os.listdir(folder_name)
        os.chdir(folder_name)
        clinvar_prePLP=pd.read_csv("Python_coderun_6-13-24_clinvar_extract_and_process_again/Python_clinvar_processed_rawdata_prePLP_7-15-24.txt", sep="\t")
        clinvar_prePLP["HGVSc_new"].to_csv("Python_coderun_6-13-24_clinvar_extract_and_process_again/Python_clinvar_processed_rawdata_prePLP_HGVScnewtoannotate_7-15-24.txt", sep="\t", index=False)
        url_CARAPI="https://reg.clinicalgenome.org/alleles?file=hgvs&fields=none+@id"
        res_CARAPI=requests.post(url_CARAPI,data=open('Python_coderun_6-13-24_clinvar_extract_and_process_again/Python_clinvar_processed_rawdata_prePLP_HGVScnewtoannotate_7-15-24.txt').read())
        CARAPI_output=pd.read_json(res_CARAPI.text)
        display(CARAPI_output)


        #join CARAPI output back to cbio dataframe first by joining to input hgvsg since expected to be in same order, and then using hgvsg column to join back to processed raw data file:
        #read HGVSg file to df
        clinvar_hgvs = pd.read_csv('Python_coderun_6-13-24_clinvar_extract_and_process_again/Python_clinvar_processed_rawdata_prePLP_HGVScnewtoannotate_7-15-24.txt', sep="\t", header=None)
        #join CAR API output to HGVSg input file by numerical index
        clinvar_HGVSg_CARAPI=clinvar_hgvs.join(CARAPI_output)
        #drop 1st row with error since 'HGVSg' header in first row
        clinvar_HGVSg_CARAPI=clinvar_HGVSg_CARAPI.drop(0)
                    
        display(clinvar_HGVSg_CARAPI)
        
        #drop rows which had 'error' output from CAR API (QC if there is an error and, df output at this step to check what and why)
        clinvar_HGVSg_CARAPI_errors=clinvar_HGVSg_CARAPI[clinvar_HGVSg_CARAPI['@id'].str.contains("error")==True] #I suppose they changed output format? so that error string does not show up in the code?
        clinvar_HGVSg_CARAPI_null=clinvar_HGVSg_CARAPI[clinvar_HGVSg_CARAPI['@id'].isnull()==True]
        
        
        clinvar_HGVSg_CARAPI.to_csv("Python_coderun_6-13-24_clinvar_extract_and_process_again/clinvar_prePLP_HGVSg_CARAPI_CAID_7-15-24.txt", sep="\t")
        
        
        cbio_CAID=pd.read_csv("Python_coderun_7-11-24_cbio_process_again/Python_cbio_processed_VEP_MANE_CAID.txt", sep="\t")
        cbio_CAID=cbio_CAID.drop_duplicates(subset=['level_0'])
        
        clinvar_HGVSg_CARAPI_nomissingCA=clinvar_HGVSg_CARAPI[~clinvar_HGVSg_CARAPI["@id"].astype(str).str.contains("_:CA")]
        clinvar_HGVSg_CARAPI_nomissingCA=clinvar_HGVSg_CARAPI_nomissingCA[clinvar_HGVSg_CARAPI_nomissingCA["@id"].isnull()==False]
        cbio_CAID_nomissingCA=cbio_CAID[~cbio_CAID["@id"].astype(str).str.contains("_:CA")]
        cbio_CAID_nomissingCA=cbio_CAID_nomissingCA[cbio_CAID_nomissingCA["@id"].isnull()==False]
        
        sharedvars= clinvar_HGVSg_CARAPI_nomissingCA.set_index("@id").join(cbio_CAID_nomissingCA.set_index("@id"), how="inner", lsuffix="clinvar", rsuffix="cbio")
        sharedvars=clinvar_prePLP.set_index("HGVSc_new").join(sharedvars.reset_index().set_index(0), how="inner", lsuffix="clinvarprePLP", rsuffix="sharedvars")
        sharedvars.to_csv("Python_coderun_6-13-24_clinvar_extract_and_process_again/Python_clinvar_cbio_tocompare.txt", sep="\t")
        
        outdf=pd.DataFrame({'Gene':[folder_name],'clinvarcount':[len(clinvar_HGVSg_CARAPI)],'cbiocount':[len(cbio_CAID)],'sharedvars':[len(sharedvars)]})
        sharedvarcounts=pd.concat([sharedvarcounts,outdf])
        
        
        print("Current directory:", os.getcwd())
        os.chdir(parentpath)
    else:
        print(f"Folder '{folder_name}' not found.")

end_time_block2=time.asctime(time.localtime())
print(end_time_block2)



## Compare ClinVar PrePLP with cBioPortal processed set

In [ ]:
# 8-16-24
start_time_block2=time.asctime(time.localtime())
print(start_time_block2)

sharedvarcounts=pd.DataFrame()

comparecounts=pd.DataFrame()


parentpath="/Users/suhasinilulla/Library/CloudStorage/Box-Box/PhD_BCM/Lulla_Plon_Lab/High_priority_gene_list/GL_S_Pipeline_1st_pass_47TSGs_May2023"
os.chdir(parentpath)


#read file with folder names and MANE Select transcript identifiers (or MANE Plus Clinical in the case of SMARCA4)
TranscriptID= pd.read_csv("TSGfolder_MANESelect_names_aalength_GeneIDs_6-13-24.txt", sep="\t")

for index, row in TranscriptID.iterrows():
    
    # Loop through each folder name
    folder_name=TranscriptID.loc[index]["Gene Symbol"]
    #if folder name exists, list all files in that folder then change directory to that folder
    if os.path.exists(folder_name) and os.path.isdir(folder_name):
        filelist=os.listdir(folder_name)
        os.chdir(folder_name)
        
        
        clinvar_prePLP_0=pd.read_csv("Python_coderun_6-13-24_clinvar_extract_and_process_again/Python_clinvar_processed_rawdata_prePLP_7-15-24.txt", sep="\t")
        
        clinvar_HGVSg_CARAPI=pd.read_csv("Python_coderun_6-13-24_clinvar_extract_and_process_again/clinvar_prePLP_HGVSg_CARAPI_CAID_7-15-24.txt", sep="\t")
        
        ReviewStatustoremove=["no classification provided","no classification for the individual variant"]
        clinvar_prePLP=clinvar_prePLP_0[~clinvar_prePLP_0["ReviewStatus"].isin(ReviewStatustoremove)]
        
        
        cbio_CAID=pd.read_csv("Python_coderun_7-11-24_cbio_process_again/Python_cbio_processed_VEP_MANE_CAID_8-16-24.txt", sep="\t")
        cbio_CAID=cbio_CAID.drop_duplicates(subset=['level_0'])
        
        clinvar_HGVSg_CARAPI_nomissingCA=clinvar_HGVSg_CARAPI[~clinvar_HGVSg_CARAPI["@id"].astype(str).str.contains("_:CA")]
        clinvar_HGVSg_CARAPI_nomissingCA=clinvar_HGVSg_CARAPI_nomissingCA[clinvar_HGVSg_CARAPI_nomissingCA["@id"].isnull()==False]
        cbio_CAID_nomissingCA=cbio_CAID[~cbio_CAID["@id"].astype(str).str.contains("_:CA")]
        cbio_CAID_nomissingCA=cbio_CAID_nomissingCA[cbio_CAID_nomissingCA["@id"].isnull()==False]
        
        sharedvars= clinvar_HGVSg_CARAPI_nomissingCA.set_index("@id").join(cbio_CAID_nomissingCA.set_index("@id"), how="inner", lsuffix="clinvar", rsuffix="cbio")
        sharedvars=clinvar_prePLP.set_index("HGVSc_new").join(sharedvars.reset_index().set_index("0"), how="inner", lsuffix="clinvarprePLP", rsuffix="sharedvars")
        sharedvars.to_csv("Python_coderun_6-13-24_clinvar_extract_and_process_again/Python_clinvar_cbio_tocompare_8-16-24.txt", sep="\t")
        
        outdf1=pd.DataFrame({'Gene':[folder_name],'clinvarcount_reviewstatusoldfilter':[len(clinvar_prePLP_0)],'clinvarcount_old':[len(clinvar_HGVSg_CARAPI)],'clinvar_prePLP_newreviewstatusfilter':[len(clinvar_prePLP)],'cbiocount':[len(cbio_CAID)],'sharedvars':[len(sharedvars)]})
        sharedvarcounts=pd.concat([sharedvarcounts,outdf1])
        
        pddf=pd.DataFrame(sharedvars["GL_first_Description"].value_counts())
        pddf=clinvarCategories.join(pddf).fillna(0)
        #pddf["GL_first_Description"]["Pathogenic"]
        compare=sharedvars
        outdf=pd.DataFrame({'Gene':[folder_name],'compare':[len(compare)],'PLP':[pddf["GL_first_Description"]["Pathogenic"]+pddf["GL_first_Description"]["Pathogenic/Likely pathogenic"]+pddf["GL_first_Description"]["Likely pathogenic"]+pddf["GL_first_Description"]["Conflicting interpretations of pathogenicity greater than or equal to 75"]],'VUS':[pddf["GL_first_Description"]["Uncertain significance"]], 'LBB':[pddf["GL_first_Description"]["Benign"]+pddf["GL_first_Description"]["Likely benign"]+pddf["GL_first_Description"]["Benign/Likely benign"]],'Conflict<75':[pddf["GL_first_Description"]["Conflicting interpretations of pathogenicity less than 75"]]})
        comparecounts=pd.concat([comparecounts,outdf])
        
        
        
        print("Current directory:", os.getcwd())
        os.chdir(parentpath)
    else:
        print(f"Folder '{folder_name}' not found.")

end_time_block2=time.asctime(time.localtime())
print(end_time_block2)



## Compare ClinVar PrePLP with COSMIC processed set

In [ ]:
#9-6-24 modified code from constructionbook2.0 here for COSMIC (didnot change names so cbio here means cosmic)
start_time_block2=time.asctime(time.localtime())
print(start_time_block2)

sharedvarcounts=pd.DataFrame()

comparecounts=pd.DataFrame()

count_total=count_total_VEPfilter=count_unique=0
parentpath="/Users/suhasinilulla/Library/CloudStorage/Box-Box/PhD_BCM/Lulla_Plon_Lab/High_priority_gene_list/GL_S_Pipeline_1st_pass_47TSGs_May2023"
os.chdir(parentpath)

clinvarCategories=pd.read_csv("ClinVarCategories_comparewithCOSMIC_10-30-23.txt", sep="\t", header=None).set_index(0)


#read file with folder names and MANE Select transcript identifiers (or MANE Plus Clinical in the case of SMARCA4)
TranscriptID= pd.read_csv("TSGfolder_MANESelect_names_aalength_GeneIDs_6-13-24.txt", sep="\t")

parentpath2="/Users/suhasinilulla/Library/CloudStorage/Box-Box/PhD_BCM/Lulla_Plon_Lab/High_priority_gene_list/COSMIC/Cosmic_MutantCensus_Tsv_v98_GRCh38"

for index, row in TranscriptID.iterrows():
    
    # Loop through each folder name
    folder_name=TranscriptID.loc[index]["Gene Symbol"]
    #if folder name exists, list all files in that folder then change directory to that folder
    if os.path.exists(folder_name) and os.path.isdir(folder_name):
        filelist=os.listdir(folder_name)
        os.chdir(folder_name)
        
        
        clinvar_prePLP_0=pd.read_csv("Python_coderun_6-13-24_clinvar_extract_and_process_again/Python_clinvar_processed_rawdata_prePLP_7-15-24.txt", sep="\t")
        
        clinvar_HGVSg_CARAPI=pd.read_csv("Python_coderun_6-13-24_clinvar_extract_and_process_again/clinvar_prePLP_HGVSg_CARAPI_CAID_7-15-24.txt", sep="\t")
        
        ReviewStatustoremove=["no classification provided","no classification for the individual variant"]
        clinvar_prePLP=clinvar_prePLP_0[~clinvar_prePLP_0["ReviewStatus"].isin(ReviewStatustoremove)]
        
        os.chdir(parentpath2)
        os.chdir(folder_name)
        
        cbio_CAID=pd.read_csv("Extract_10-2-23/COSMIC_processed_VEP_MANE_CAID_9-6-24.txt", sep="\t")
        count_total=count_total+len(cbio_CAID)
        HGVScremove=["-"]
        cbio_VEP_cancertype=cbio_CAID[~cbio_CAID["HGVSc"].isin(HGVScremove)==True]
        
        
        VEP_categories_tofilter_list=["synonymous_variant","5_prime_UTR_variant", "3_prime_UTR_variant", "non_coding_transcript_exon_variant", "intron_variant", "non_coding_transcript_variant", "upstream_gene_variant", "downstream_gene_variant", "regulatory_region_ablation", "regulatory_region_amplification", "regulatory_region_variant", "intergenic_variant", "splice_region_variant"]
        cbio_VEP_cancertype=cbio_VEP_cancertype[~cbio_VEP_cancertype["collapsed_Consequence"].isin(VEP_categories_tofilter_list)]

        count_total_VEPfilter=count_total_VEPfilter+len(cbio_VEP_cancertype)
        cbio_CAID=cbio_VEP_cancertype.drop_duplicates(subset=['level_0'])
        count_unique=count_unique+len(cbio_CAID)
        
        clinvar_HGVSg_CARAPI_nomissingCA=clinvar_HGVSg_CARAPI[~clinvar_HGVSg_CARAPI["@id"].astype(str).str.contains("_:CA")]
        clinvar_HGVSg_CARAPI_nomissingCA=clinvar_HGVSg_CARAPI_nomissingCA[clinvar_HGVSg_CARAPI_nomissingCA["@id"].isnull()==False]
        cbio_CAID_nomissingCA=cbio_CAID[~cbio_CAID["@id"].astype(str).str.contains("_:CA")]
        cbio_CAID_nomissingCA=cbio_CAID_nomissingCA[cbio_CAID_nomissingCA["@id"].isnull()==False]
        
        sharedvars= clinvar_HGVSg_CARAPI_nomissingCA.set_index("@id").join(cbio_CAID_nomissingCA.set_index("@id"), how="inner", lsuffix="clinvar", rsuffix="COSMIC")
        sharedvars=clinvar_prePLP.set_index("HGVSc_new").join(sharedvars.reset_index().set_index("0clinvar"), how="inner", lsuffix="clinvarprePLP", rsuffix="sharedvars")
        #sharedvars.to_csv("Extract_10-2-23/Python_clinvar_COSMIC_tocompare_9-6-24.txt", sep="\t")
        
        outdf1=pd.DataFrame({'Gene':[folder_name],'clinvarcount_reviewstatusoldfilter':[len(clinvar_prePLP_0)],'clinvarcount_old':[len(clinvar_HGVSg_CARAPI)],'clinvar_prePLP_newreviewstatusfilter':[len(clinvar_prePLP)],'cbiocount':[len(cbio_CAID)],'sharedvars':[len(sharedvars)]})
        sharedvarcounts=pd.concat([sharedvarcounts,outdf1])
        
        pddf=pd.DataFrame(sharedvars["GL_first_Description"].value_counts())
        pddf=clinvarCategories.join(pddf).fillna(0)
        #pddf["GL_first_Description"]["Pathogenic"]
        compare=sharedvars
        outdf=pd.DataFrame({'Gene':[folder_name],'compare':[len(compare)],'PLP':[pddf["count"]["Pathogenic"]+pddf["count"]["Pathogenic/Likely pathogenic"]+pddf["count"]["Likely pathogenic"]+pddf["count"]["Conflicting interpretations of pathogenicity greater than or equal to 75"]],'VUS':[pddf["count"]["Uncertain significance"]], 'LBB':[pddf["count"]["Benign"]+pddf["count"]["Likely benign"]+pddf["count"]["Benign/Likely benign"]],'Conflict<75':[pddf["count"]["Conflicting interpretations of pathogenicity less than 75"]]})
        comparecounts=pd.concat([comparecounts,outdf])
        
        
        
        print("Current directory:", os.getcwd())
        os.chdir(parentpath)
    else:
        print(f"Folder '{folder_name}' not found.")

end_time_block2=time.asctime(time.localtime())
print(end_time_block2)



In [32]:
print(count_total)
print(count_total_VEPfilter)
print(count_unique)


58468
57147
15300


In [34]:
comparecounts.to_csv("COSMIC/Cosmic_MutantCensus_Tsv_v98_GRCh38/comparecounts_forfigS4_11-25-25.txt", sep="\t")

In [36]:
comparecounts.sum()

Gene           WT1VHLTSC2TSC1TP53SUFUSTK11SMARCB1SMARCA4SMAD4...
compare                                                     4941
PLP                                                       4080.0
VUS                                                        584.0
LBB                                                         33.0
Conflict<75                                                243.0
dtype: object